In [8]:
import numpy as np
import pandas as pd

import geopy
from geopy.geocoders import Nominatim

from math import radians, cos, sin, asin, sqrt
import googlemaps
from datetime import datetime

import json

# Cleaning Data

In [3]:
yerevan_actual_raw = pd.read_excel('./data/Yerevan actual.xlsx')

## filtering out duplicated in id 

In [67]:
yerevan_actual_raw = yerevan_actual_raw.drop(["OBJECTID", "level_0", "index"], axis = 1)
duplicates = yerevan_actual_raw.duplicated(subset=['id'])
yerevan_actual = yerevan_actual_raw[~duplicates].drop_duplicates(subset=['id'])
yerevan_actual = yerevan_actual.reset_index()

## Ceategorizing Regions

In [ ]:
data = yerevan_actual.copy()

In [ ]:
geolocator = Nominatim(user_agent="my_application")

count = 0
yerevan_districts = pd.DataFrame(columns=['x', 'y', 'district'])

import pandas as pd

# Create an empty DataFrame to store the results


def get_district(lat, lng):
    # Reverse geocode the coordinates to get the address
    location = geolocator.reverse(f"{lat}, {lng}", timeout=None)
    # Extract the district information from the address
    address = location.raw['address']
    district = address.get('suburb', address.get('city_district'))
    return district

# Loop through each row of the data and calculate the district for each row
# for i, row in data.iterrows():
#     # Calculate the district for the current row
#     district = get_district(row['x'], row['y'])
#     count = count + 1
#     data.drop(index=data.index[0], axis=0, inplace=True)
#     # Append the current row to the new DataFrame
#     yerevan_districts = yerevan_districts.append({'x': row['x'], 'y': row['y'], 'district': district}, ignore_index=True)

In [ ]:
# Save the results to a CSV file
# yerevan_actual["district"] = yerevan_districts["district"]
# yerevan_actual.to_excel("Yerevan.xlsx", index=False)

In [5]:
yerevan_actual = pd.read_excel('./data/Yerevan.xlsx')

## Metro Distance Calculation

In [ ]:
h1 = pd.read_excel('./data/Yerevan.xlsx')
houses = h1.copy() # houses is used for testing

# Load the houses and stations tables into pandas dataframes

stations = pd.DataFrame({
    'station_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'station_name': ['Barekamutyun', 'Baghramyan', 'Yeritasardakan', 'Republic Square', 'Zoravar Andranik', 'Sasuntsi Davit', 'Gortsaranain','Shengavit', 'Charbakh', 'Garegin Nzhdeh Square'],
    'x': [40.197446,40.1918167,40.1873081, 40.1785331, 40.1699272, 40.1552243, 40.1410611, 40.145224, 40.1405941, 40.1504761],
    'y': [44.493467,44.5038091, 44.5191743, 44.5134433, 44.5112603, 44.5058011, 44.4964173, 44.48483, 44.4685333, 44.4816183]
})

In [ ]:
# Define a function to calculate the Haversine distance between two points
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    r = 6371 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

def get_walking_distance(start_lon, start_lat, end_lon, end_lat):
    # Set up the Google Maps API client
    gmaps = googlemaps.Client(key='***REMOVED-GOOGLE-API-KEY***')

    # Define the start and end coordinates
    start = (start_lat, start_lon)
    end = (end_lat, end_lon)

    # Make the API request
    now = datetime.now()
    directions = gmaps.directions(start, end, mode='walking', departure_time=now)

    # Extract the distance from the response
    distance = directions[0]['legs'][0]['distance']['value']

    # Return the distance in meters
    return distance

#### calculating distance from every house to every metro station and taking shortest path

In [ ]:
# Loop through each house and station to find the shortest distance using Haversine
#distances = []
# station_ids = []
# station_names = []
# for house_idx, house in houses.iterrows():
#     shortest_distance = float('inf')
#     nearest_station_name = ''
#     for station_idx, station in stations.iterrows():
#         distance = haversine(house['y'], house['x'], station['y'], station['x'])
#         if distance < shortest_distance:
#             shortest_distance = distance
#             nearest_station_id = station['station_id']
#             nearest_station_name = station['station_name']
#     #distances.append(shortest_distance)
#     station_ids.append(nearest_station_id)
#     station_names.append(nearest_station_name)

# # Add the new columns to the houses table
# #houses['shortest_distance'] = distances
# houses['nearest_station_id'] = station_ids
# houses['nearest_station_name'] = station_names
# houses

#### we have closest metro station using haversine distance, now we calculate the walking distance

In [19]:
# h1["closest_metro"] = houses["nearest_station_name"]
# distances = []

# # Using Google Maps API to calculate the walking distance
# for i, house in houses.iterrows():
#     distance = get_walking_distance(house['y'], house['x'], stations['y'][house['nearest_station_id']-1], stations['x'][house['nearest_station_id']-1])
#     distances.append(distance)
#     houses.drop(index=houses.index[0], axis=0, inplace=True)
# #houses["walk_distance"] = distances

# h1["walking_distance_to_metro(m)"] = distances

# Overwriting Yervan.xlsx file with the new metro and walking distance columns
# h1.to_excel("Yerevan.xlsx", index=False)

In [17]:
yerevan_actual = pd.read_excel('./data/Yerevan.xlsx')

### rent and sales real-estate

In [23]:
sale = yerevan_actual.loc[(yerevan_actual["rent_or_sale"]=="sale appartments") | (yerevan_actual["rent_or_sale"]=="sale houses")]
rent = yerevan_actual.loc[(yerevan_actual["rent_or_sale"]=="rent houses") | (yerevan_actual["rent_or_sale"]=="rent appartments")]